In [1]:
import json
from collections import Counter
from typing import Literal

from pydantic import BaseModel, Field, ValidationError
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
biletler = [
    "Bu ay kredi kartımdan iki kez çekim yapılmış, hemen iade istiyorum!",
    "Faturamdaki tutar paketimle uyuşmuyor, açıklar mısınız?",
    "Uygulama her açtığımda çöküyor, iki gündür çalışamıyorum.",
    "Dosya yüklerken sürekli 500 hatası alıyorum.",
    "Şifremi unuttum ve sıfırlama maili hiç gelmiyor.",
    "Hesabımı başka bir e-posta adresine taşımak istiyorum.",
    "Merhaba, ürününüzde karanlık mod ne zaman gelecek?",
    "Ekibinize teşekkür etmek istedim, destek çok hızlıydı.",
    "Ödeme yaptım ama hesabım hâlâ açılmadı, ne yapmalıyım?",
    "Giriş yapamıyorum, hata kodu 403 veriyor.",
    "Yıllık aboneliği iptal edersem ücret iadesi olur mu?",
    "Kurumsal toplantı için sizinle görüşmek istiyoruz.",
]

insan_kategorileri = [
    "fatura", "fatura", "teknik", "teknik", "hesap", "hesap",
    "diger", "diger", "fatura", "teknik", "fatura", "diger",
]

print(len(biletler), len(insan_kategorileri))

12 12


In [4]:
class BiletAnaliz(BaseModel):
    kategori: Literal["fatura", "teknik", "hesap", "diger"] = Field(description="Kategori değeri")
    oncelik: int = Field(ge=1, le=5, description="1 = önemsiz ... 5 = acil müdahale")
    duygu: Literal["sakin", "sinirli"] = Field(description="Duygu durumu")

In [6]:
def build_prompt(bilet):
    prompt = f"""
    Sen bir destek bileti sınıflandırıcısın. Aşağıdaki bileti analiz et ve sadece geçerli bir JSON döndür. Başka hiçbir açıklama ekleme.

    {{
        "kategori": "fatura | teknik | hesap | diger",
        "oncelik: 1 ile 5 arası tam sayı,
        "duygu": "sakin | sinirli"
    }}

    Kategori kuralları:
- fatura: para, ödeme, iade, ücret, abonelik konuları
- teknik: ... (sen yaz)
- hesap: ... (sen yaz)
- diger: ... (sen yaz)
- Bilet hem ödemeyi hem hesabı ilgilendiriyorsa: ... (sen yaz)

Öncelik ölçeği:
- 5: ... (sen yaz)
- 3: ... (sen yaz)
- 1: ... (sen yaz)

Bilet:
{bilet}
"""

    return prompt.strip()

In [7]:
def sahte_llm(prompt):
    # Sadece "Bilet:" sonrasını al, kural satırları karışmasın
    bilet = prompt.split("Bilet:")[-1].strip().lower()

    # Kategori: yukarıdan aşağı ilk eşleşen kazanır
    if any(k in bilet for k in ["çekim", "fatura", "iade", "ödeme", "ücret"]):
        kategori = "fatura"
    elif any(k in bilet for k in ["çök", "hata", "yüklerken"]):
        kategori = "teknik"
    elif any(k in bilet for k in ["şifre", "hesab", "giriş"]):
        kategori = "hesap"
    else:
        kategori = "diger"

    # Duygu: ünlem veya "hemen" varsa sinirli
    if "!" in bilet or "hemen" in bilet:
        duygu = "sinirli"
    else:
        duygu = "sakin"

    # Öncelik: kategoriye göre sabit tablo
    oncelik_tablosu = {"fatura": 4, "teknik": 3, "hesap": 3, "diger": 1}
    oncelik = oncelik_tablosu[kategori]

    return json.dumps({"kategori": kategori, "oncelik": oncelik, "duygu": duygu})

In [8]:
for b in biletler[:4]:
    cevap = sahte_llm(build_prompt(b))
    analiz = BiletAnaliz.model_validate_json(cevap)
    print(b[:45].ljust(47), "->", analiz.model_dump())

Bu ay kredi kartımdan iki kez çekim yapılmış,   -> {'kategori': 'fatura', 'oncelik': 4, 'duygu': 'sinirli'}
Faturamdaki tutar paketimle uyuşmuyor, açıkla   -> {'kategori': 'fatura', 'oncelik': 4, 'duygu': 'sakin'}
Uygulama her açtığımda çöküyor, iki gündür ça   -> {'kategori': 'teknik', 'oncelik': 3, 'duygu': 'sakin'}
Dosya yüklerken sürekli 500 hatası alıyorum.    -> {'kategori': 'teknik', 'oncelik': 3, 'duygu': 'sakin'}


In [9]:
def siniflandir(bilet, llm, deneme=2):
    prompt = build_prompt(bilet)

    for i in range(deneme):
        cevap = llm(prompt)
        try:
            
            return  BiletAnaliz.model_validate_json(cevap).model_dump()
        except ValidationError:
            print(f"Deneme {i+1} başarısız, tekrar deneniyor...")

    return None   

In [10]:
def toplu_siniflandir(biletler, llm):
    sonuclar = []
    for bilet in biletler:
        kayit = siniflandir(bilet, llm)
        sonuclar.append(kayit)
    return sonuclar


kayitlar = toplu_siniflandir(biletler, sahte_llm)

llm_kategorileri = [k["kategori"] for k in kayitlar]
acc = accuracy_score(insan_kategorileri, llm_kategorileri)

print("Accuracy:", round(acc, 2))

for b, insan, llm_k in zip(biletler, insan_kategorileri, llm_kategorileri):
    if insan != llm_k:
        print(f"- {b[:50]} | insan: {insan} | llm: {llm_k}")

Accuracy: 1.0
